# 06 — Rolling monthly retrain vs frozen model

Does retraining the model monthly improve realized ROI on the Polymarket-matched test period?

## Experimental setup

For each month M in the test window (2024-04 → 2026-03):
1. **Rolling**: retrain v3 CatBoost on all Kaggle fights with `date < first_day_of_M`. Skill features parquet already covers all dates leakage-free (built via monthly walk-forward). Same recipe as `v3_full2000_trainval` (full 2000 iters, no early stopping, seed=0, symmetry augmentation, recency reference = end of previous month).
2. **Frozen**: use `v3_full2000_trainval` (trained once on train+val through 2023-12-31) for every month.
3. **Bankroll carries over** from previous month within each account.
4. Three accounts simulated, $300 starting each:
   - **A**: 10%-K + 10% cap, real model
   - **B**: ¼-K + no cap, real model
   - **C**: ¼-K + no cap, corrupted-skill variant
5. Polymarket prices, 2% fee, 3% edge threshold.

Raw results were produced by [scripts/rolling_monthly_backtest.py](../scripts/rolling_monthly_backtest.py) and live in [artifacts/metrics/rolling_monthly_backtest.json](../artifacts/metrics/rolling_monthly_backtest.json).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent

result = json.loads((ROOT / 'artifacts/metrics/rolling_monthly_backtest.json').read_text())
monthly = pd.DataFrame(result['monthly'])
monthly['month'] = pd.to_datetime(monthly['month'] + '-01')
monthly = monthly.sort_values('month').reset_index(drop=True)
print(f'Months covered: {len(monthly)}')
print(f'Range: {monthly["month"].min().strftime("%Y-%m")} → {monthly["month"].max().strftime("%Y-%m")}')
print(f'Total matched fights bet on: {monthly["n_fights"].sum()}')

## Headline numbers

In [ ]:
headline = []
for tag in ['A', 'B', 'C']:
    cfg = next(a for a in result['config']['accounts'] if a['tag'] == tag)
    rolling = result['final_bankrolls']['rolling'][tag]
    frozen  = result['final_bankrolls']['frozen'][tag]
    cap_label = 'no cap' if cfg['cap'] >= 1 else f"{int(cfg['cap']*100)}% cap"
    headline.append({
        'account': tag,
        'model':   cfg['src'],
        'config':  f"{int(cfg['kelly']*100)}%-K + {cap_label}",
        'rolling': rolling,
        'frozen':  frozen,
        'ratio':   rolling / frozen,
    })
headline_df = pd.DataFrame(headline)
headline_df.style.format({
    'rolling': '${:,.2f}',
    'frozen':  '${:,.2f}',
    'ratio':   '{:.2f}×',
})

## Per-month bankroll trajectories — rolling vs frozen

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
colors = {'rolling': '#1a9641', 'frozen': '#d7191c'}

for row, tag in enumerate(['A', 'B', 'C']):
    for col, scale in enumerate(['linear', 'log']):
        ax = axes[row, col]
        ax.plot(monthly['month'], monthly[f'{tag}_rolling_end'],
                color=colors['rolling'], linewidth=2, marker='o', markersize=4,
                label=f'Rolling (monthly retrain) → ${monthly[f"{tag}_rolling_end"].iloc[-1]:,.0f}')
        ax.plot(monthly['month'], monthly[f'{tag}_frozen_end'],
                color=colors['frozen'], linewidth=2, marker='s', markersize=4,
                label=f'Frozen → ${monthly[f"{tag}_frozen_end"].iloc[-1]:,.0f}')
        ax.axhline(300, color='black', linestyle='-', alpha=0.4, linewidth=0.8, label='$300 start')
        ax.set_yscale(scale)
        ax.set_ylabel(f'Bankroll ($) — {scale}')
        ax.set_title(f'Account {tag} — {scale} scale')
        ax.legend(loc='upper left' if scale == 'linear' else 'lower right', fontsize=9)
        ax.grid(alpha=0.3, which='both')

axes[2, 0].set_xlabel('Month')
axes[2, 1].set_xlabel('Month')
plt.tight_layout()
plt.show()

## Per-month table — bankroll at end of each month, both regimes

In [ ]:
display_cols = ['month', 'n_fights',
                'A_rolling_end', 'A_frozen_end',
                'B_rolling_end', 'B_frozen_end',
                'C_rolling_end', 'C_frozen_end']
tbl = monthly[display_cols].copy()
tbl['month'] = tbl['month'].dt.strftime('%Y-%m')
tbl.style.format({c: '${:,.2f}' for c in display_cols if c not in ('month','n_fights')})

## Per-month P&L delta — does rolling outperform frozen each month?

If rolling is genuinely better, we expect more months with positive `(rolling_pnl − frozen_pnl)` than negative. If it's just one or two lucky months, we'd see the gap concentrated.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, tag in zip(axes, ['A', 'B', 'C']):
    delta = monthly[f'{tag}_rolling_pnl'] - monthly[f'{tag}_frozen_pnl']
    colors_pos = ['#1a9641' if x >= 0 else '#d7191c' for x in delta]
    ax.bar(monthly['month'], delta, color=colors_pos, width=22, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Rolling P&L − Frozen P&L ($)')
    ax.set_title(f'Account {tag} — per-month outperformance of rolling vs frozen  '
                 f'({(delta > 0).sum()} months rolling won, {(delta < 0).sum()} months frozen won)')
    ax.set_yscale('symlog', linthresh=10)
    ax.grid(alpha=0.3, axis='y')
axes[-1].set_xlabel('Month')
plt.tight_layout()
plt.show()

# Summary count
print('Months where ROLLING outperformed FROZEN:')
for tag in ['A','B','C']:
    delta = monthly[f'{tag}_rolling_pnl'] - monthly[f'{tag}_frozen_pnl']
    print(f'  Account {tag}: {(delta > 0).sum()}/{len(delta)} months  '
          f'(mean delta = ${delta.mean():+,.2f}, median = ${delta.median():+,.2f})')

## Running ratio (rolling_bankroll / frozen_bankroll) over time

Above 1.0 means rolling has accumulated more. Below 1.0 means frozen leads.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {'A': '#1a9641', 'B': '#1f78b4', 'C': '#d7191c'}
for tag in ['A','B','C']:
    ratio = monthly[f'{tag}_rolling_end'] / monthly[f'{tag}_frozen_end']
    ax.plot(monthly['month'], ratio, color=colors[tag], linewidth=2,
            marker='o', markersize=4,
            label=f'Account {tag} — final ratio {ratio.iloc[-1]:.2f}×')
ax.axhline(1.0, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
ax.set_yscale('log')
ax.set_ylim(0.05, 5)
ax.set_ylabel('Rolling bankroll / Frozen bankroll  (log scale)')
ax.set_xlabel('Month')
ax.set_title('Rolling/Frozen ratio over time — above 1 = rolling wins, below 1 = frozen wins')
ax.legend(loc='upper left')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Per-bet geometric growth rate — frozen vs rolling

Compounds out the bankroll difference and gives a cleaner per-bet rate comparison.

In [ ]:
per_bet_rows = []
total_bets = monthly['A_n_bets'].sum()  # n_bets is per-account but same for A/B since same model
for tag in ['A','B','C']:
    n = monthly[f'{tag}_n_bets'].sum()
    rolling_final = result['final_bankrolls']['rolling'][tag] / 300
    frozen_final  = result['final_bankrolls']['frozen'][tag] / 300
    rolling_gm = rolling_final ** (1 / max(n, 1))
    frozen_gm  = frozen_final  ** (1 / max(n, 1))
    per_bet_rows.append({
        'account': tag,
        'n_bets': int(n),
        'rolling_growth': rolling_final,
        'frozen_growth':  frozen_final,
        'rolling_per_bet': rolling_gm,
        'frozen_per_bet':  frozen_gm,
        'delta_per_bet':  (rolling_gm - frozen_gm) * 100,  # in percentage points
    })
per_bet_df = pd.DataFrame(per_bet_rows)
per_bet_df.style.format({
    'rolling_growth': '{:.1f}×',
    'frozen_growth':  '{:.1f}×',
    'rolling_per_bet': '{:.6f}',
    'frozen_per_bet':  '{:.6f}',
    'delta_per_bet':   '{:+.4f} pp',
})

## Takeaways

1. **For the real model (Accounts A and B): monthly retraining genuinely helps.** Per-bet geometric return increases by ~0.05 percentage points (1.0121 → 1.0126 for Account B). Compounded over 600+ bets, that delivers a 1.40× final-bankroll improvement ($484k → $677k on Account B from $300 starting). The mechanism is straightforward: monthly retrain incorporates 25-50 fresh fights and their walk-forward skill features, capturing recent meta drift the 2023-frozen model misses.

2. **For the corrupted variant (Account C): monthly retraining destroys the advantage.** Frozen Account C compounded to $7.5M on Polymarket; the rolling-retrained version only reached $565k — a 13× gap. This confirms what we suspected in the earlier analysis: Account C's outsized final bankroll under the frozen model is a *specific tree-structure quirk*, not portable signal. Each new monthly retrain creates a different (also quirk-shaped, but differently-quirk-shaped) corrupted variant, and the original lucky bet-sequence-dodging pattern is destroyed.

3. **Even rolling-corrupted underperforms rolling-real.** Account B rolling ($677k) > Account C rolling ($565k). So once you remove the frozen-corrupted lottery effect, the corrupted variant is strictly worse than the real model at the same Kelly sizing. There's no signal advantage to NaN'ing the skill features; the only "benefit" was variance reduction on a specific test sequence with a specific frozen model.

4. **Most months, rolling does NOT outperform frozen by much** — the gains are concentrated. Looking at per-month P&L deltas, rolling wins more months than it loses for the real model, but the magnitude of individual months is modest. The big-picture 1.40× advantage is the product of many small per-bet improvements compounding over 600+ bets. This is what you'd expect from a real, small per-bet edge.

5. **Implications for DEPLOY.md:**
   - **Account A**: 10%-K + 10% cap, real model, **retrain monthly**. ~1.1× improvement.
   - **Account B**: ¼-K + no cap, real model, **retrain monthly**. ~1.4× improvement.
   - **Account C**: **drop or replace.** The frozen version's $7.5M outcome is unreproducible — it was specific to the trained-once `v3_full2000_no_skill_corrupted_trainval` artifact and the particular bet sequence of test. Retraining ruins the property. Treat this as confirmation that the corrupted-variant story was sequence-luck, not edge.

6. **Operational cadence.** Retrain once per month — the simplest schedule that delivers ~all the upside. Skill features parquet rebuild is the slow part (~20 min on CPU per refresh), but it's the same monthly walk-forward we already run. The CatBoost retrain itself is ~30 seconds.

7. **Caveat.** This is one realized 24-month window. The per-bet rate improvement (+0.005 pp) is small enough that a different period could plausibly show frozen winning or both being roughly tied. Use the rolling improvement as a positive signal, not as a precise expectation.